In [5]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
import os
import numpy as np
import imageio
from tensorflow.keras import layers

LATENT_DIM = 256
IMG_HEIGHT = 128
IMG_WIDTH = 128

def make_generator():
    model = tf.keras.Sequential([
        layers.Dense(8*8*512, use_bias=False, input_shape=(LATENT_DIM,)),
        layers.BatchNormalization(),
        layers.LeakyReLU(alpha=0.2),
        layers.Reshape((8, 8, 512)),
        layers.Conv2DTranspose(256, (5,5), strides=2, padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(alpha=0.2),
        layers.Conv2DTranspose(128, (5,5), strides=2, padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(alpha=0.2),
        layers.Conv2DTranspose(64, (5,5), strides=2, padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(alpha=0.2),
        layers.Conv2DTranspose(32, (5,5), strides=2, padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(alpha=0.2),
        layers.Conv2DTranspose(1, (5,5), strides=1, padding='same', use_bias=False, activation='tanh')
    ])
    return model

# Paths
drive_path = "/content/drive/MyDrive"
save_dir = os.path.join(drive_path, "GAN_Models")
output_dir = os.path.join(drive_path, "Generated_Xrays_Single")
os.makedirs(output_dir, exist_ok=True)

# Initialize and load generator
generator = make_generator()
checkpoint_prefix = os.path.join(save_dir, "ckpt-71")
checkpoint = tf.train.Checkpoint(generator=generator)
status = checkpoint.restore(checkpoint_prefix)
status.expect_partial()  # Suppress warnings for missing optimizer/discriminator

# Verify checkpoint load
print("Checkpoint restoration status:")
status.assert_existing_objects_matched()  # Ensure generator weights are loaded

# Generate images
num_images = 400
noise = tf.random.normal([num_images, LATENT_DIM])
generated_images = generator(noise, training=False)

# Save images
for i in range(num_images):
    img_array = generated_images[i].numpy()
    img_array = (img_array * 127.5 + 127.5).astype(np.uint8).squeeze()
    image_path = os.path.join(output_dir, f'generated_xray_{i+1}.png')
    imageio.imwrite(image_path, img_array)

print(f"\nGenerated images saved to: {output_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint restoration status:

Generated images saved to: /content/drive/MyDrive/Generated_Xrays_Single
